# The Knight's Maze

### Lab 1 · Search

A knight (K) is tasked to retrieve the treasure (G) hidden in the dungeon without being caught by a dragon (D). 

You are tasked in writing the movement algorithm for the Knight agent. To pass the exercise, the BFS and Dijkstra algorithm that you already know are enough. Additional difficulty levels are provided if you want to enjoy a challenge. 

You can work on this page, on Google Colab, or locally (see *Running the code*). In any case, upload a notebook back on the platform once you are finished. 

### AI Policy: 

While AI can be used, I do not recommended using it for directly writing the code to solve the exercises. [Preliminary research](https://arxiv.org/abs/2506.08872) shows reduced EEG connectivity when offloading to AI, inducing poorer retention. 

I recommend to try solving the exercise without AI, and interrogate AI on how to understand specific portions of the code, or how to use specific functions in python, rather than actually writing the code. 

The goal always not to solve the exercise, but to learn. When you will start working, you will probably be using mostly AI for writing code, - but if you can't understand the code that AI is writing, it will be impossible to debug or to understand the implication of what you are doing. 

***We ask you to disclose at the start of the notebook the kind of AI assistance that was used to solve the exercises.***

### Prerequisites.

This exercise is meant as a recap for both *Object Oriented Programming* in Python and for the graph navigation algorithms. You can recap OOP [here](https://realpython.com/python3-object-oriented-programming/), but you also will need [Abstract Base Classes](https://docs.python.org/3/library/abc.html) and [Dataclasses](https://docs.python.org/3/library/dataclasses.html), which are widely used when programming AI. 

In particular, you should be knowing [BFS](https://it.wikipedia.org/wiki/Ricerca_in_ampiezza) and [Dijkstra's](https://en.wikipedia.org/wiki/Dijkstra%27s_algorithm). If you don't remember them, use this as an opportunity to recap them. More advanced versions of them will be used in the course. 

### Running the code.

You can do the exercise directly in the platform, as it download python in the browser through [pyodide](https://pyodide.org/en/stable/), on Google Colab or locally. 

If you chose to do it locally, I recommend running it inside a dedicated virtual environment using [uv](https://docs.astral.sh/uv/getting-started/installation/). Running the following command after installing uv both creates a virtual environment, a package.json file and sets up git in the folder. 

```bash
uv init

## the following installs jupyter in the local venv
uv add ipykernel 
``` 
Further syncing up the folder with GitHub is also recommended. 


## The exercise

The dungeon is a grid of cells drawn in ASCII. In the initial version, the knight starts top-left, the treasure is bottom-right and the dragon starts in the middle. The base version is a 10*10 connected graph, with no walls. Additional versions have walls, like in this example: 

    █████████████████
    █ K .   █   .   █
    █   ·   █   ·   █
    █   .   █ D .   █
    █   █████   ·   █
    █   .   .   . G █
    █████████████████

**Rules.** Each turn the knight moves first, then the dragon. A move is a step (in the four directions, no diagonal moves) to a neighbouring cell with no wall in between, or staying still. The knight wins when it reaches the treasure. It loses if it ends up on the same cell as the dragon, or if 200 turns pass without reaching the treasure. 


### The Notebook.

The notebook has a lot of code to draw the maze and to evaluate the runs. You don't have to read them and can simply run those. 

You should first familiarise with the maze, and be able to make the knight navigate it statically, by computing both BFS and Dijkstra on the static maze. 

Once you are able to draw paths across the maze, you can add those strategies to the knight. 


**Levels.** Levels 1 and 2 are required to pass. The others are optional and open-ended.

| Level | Dragon | Algorithm | To pass |
|---|---|---|---|
| 1 | `StaticDragon`: sleeps where it stands | BFS, with the dragon's cell treated as a wall | ≥ 95 % of 50 random mazes |
| 2 | `GuardDragon`: sleeps, but hunts you if you come within 3 steps of its lair | Dijkstra, with a cost that grows near the dragon | ≥ 75 % of 50 random mazes |
| 2.5 | `GreedyDragon` | your choice | 90 |
| 3 | `ChaserDragon` at half or full speed, on mazes with loops (`"density": 0.5`) | your choice | 70 |


**Testing** The last cell allows you to both have a visual experience of the maze, or to run evaluate on 50 mazes.


In [34]:
"""The maze: a rows x cols grid graph, drawn as text.

Each cell (row, col) is a node, and two neighbouring cells are connected
unless there is a wall between them. A new maze has no walls. You can add
them through the `walls` parameter, with add_wall(), or with random_walls().

    maze = Maze(4, 6, walls=[((0, 0), (0, 1)), ((1, 2), (2, 2))])
    print(maze)

Walls are drawn as solid blocks, and cells are separated by faint dots so
the grid stays visible even with no walls. Paths are drawn as arrows
pointing to the next cell: single arrows for the knight, double arrows
for the dragon, and a cross where both passed.
"""
import random

DIRECTIONS = ((-1, 0), (1, 0), (0, -1), (0, 1))  # up, down, left, right

WALL, DOT, BOTH = "█", "·", "✕"
ARROWS = {
    "knight": {(-1, 0): "↑", (1, 0): "↓", (0, -1): "←", (0, 1): "→"},
    "dragon": {(-1, 0): "⇑", (1, 0): "⇓", (0, -1): "⇐", (0, 1): "⇒"},
}


def manhattan(a, b):
    """Grid distance between two cells, ignoring walls."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def path_arrows(path, arrows):
    """{cell: arrow pointing to the next cell on the path}. A cell visited
    more than once keeps its latest arrow; the last cell gets none."""
    out = {}
    for a, b in zip(path, path[1:]):
        if a != b:
            out[tuple(a)] = arrows[(b[0] - a[0], b[1] - a[1])]
    return out


class Maze:
    def __init__(self, rows=10, cols=None, walls=()):
        """A rows x cols maze (square if cols is omitted).

        `walls` is a list of pairs of neighbouring cells with a wall between
        them, e.g. [((0, 0), (0, 1))] puts a wall between the two top-left
        cells.
        """
        self.rows = rows
        self.cols = rows if cols is None else cols
        self.walls = set()
        for a, b in walls:
            self.add_wall(a, b)

    # ---------- cells and walls ----------

    def in_bounds(self, cell):
        r, c = cell
        return 0 <= r < self.rows and 0 <= c < self.cols

    def cells(self):
        """All cells, row by row."""
        return [(r, c) for r in range(self.rows) for c in range(self.cols)]

    def _edge(self, a, b):
        a, b = tuple(a), tuple(b)
        if not (self.in_bounds(a) and self.in_bounds(b)) or manhattan(a, b) != 1:
            raise ValueError(
                f"a wall must separate two neighbouring cells inside the "
                f"{self.rows}x{self.cols} maze, got {a} and {b}")
        return frozenset((a, b))

    def add_wall(self, a, b):
        self.walls.add(self._edge(a, b))

    def is_open(self, a, b):
        """True if you can step directly from cell a to cell b."""
        return (self.in_bounds(a) and self.in_bounds(b) and manhattan(a, b) == 1
                and frozenset((a, b)) not in self.walls)

    # ---------- graph view ----------

    def neighbours(self, cell):
        """Cells you can step to from `cell` (up, down, left, right)."""
        r, c = cell
        return [(r + dr, c + dc) for dr, dc in DIRECTIONS
                if self.is_open(cell, (r + dr, c + dc))]

    def adjacency(self):
        """The whole maze as an adjacency list: {cell: [neighbours]}."""
        return {cell: self.neighbours(cell) for cell in self.cells()}

    # ---------- drawing ----------

    def _wall_above(self, r, c):
        """Wall on the top edge of cell (r, c); r == rows is the bottom border."""
        return r in (0, self.rows) or not self.is_open((r - 1, c), (r, c))

    def _wall_left(self, r, c):
        """Wall on the left edge of cell (r, c); c == cols is the right border."""
        return c in (0, self.cols) or not self.is_open((r, c - 1), (r, c))

    def _wall_at_corner(self, r, c):
        """True if any wall segment touches the corner point (r, c)."""
        return ((c < self.cols and self._wall_above(r, c))
                or (c > 0 and self._wall_above(r, c - 1))
                or (r < self.rows and self._wall_left(r, c))
                or (r > 0 and self._wall_left(r - 1, c)))

    def render(self, marks=None, knight_path=(), dragon_path=()):
        """The maze as text.

        `marks` maps cells to single characters, e.g. {(0, 0): "K"}, and is
        drawn on top of the paths; `knight_path` and `dragon_path` are
        lists of cells, drawn as arrows.
        """
        km = path_arrows(list(knight_path), ARROWS["knight"])
        dm = path_arrows(list(dragon_path), ARROWS["dragon"])
        symbols = {cell: BOTH if cell in km and cell in dm else km.get(cell) or dm[cell]
                   for cell in set(km) | set(dm)}
        symbols.update({tuple(cell): ch for cell, ch in (marks or {}).items()})

        lines = []
        for r in range(self.rows + 1):
            line = ""
            for c in range(self.cols + 1):
                line += WALL if self._wall_at_corner(r, c) else DOT
                if c < self.cols:
                    line += WALL * 3 if self._wall_above(r, c) else "   "
            lines.append(line)
            if r < self.rows:
                line = ""
                for c in range(self.cols + 1):
                    line += WALL if self._wall_left(r, c) else " "
                    if c < self.cols:
                        line += " " + symbols.get((r, c), " ") + " "
                lines.append(line)
        return "\n".join(lines)

    def __str__(self):
        return self.render()

    __repr__ = __str__  # Jupyter shows this when a maze ends a cell


def random_walls(rows, cols=None, density=0.8, seed=None):
    """Walls for a random maze in which every cell can be reached.

    It first builds a "perfect" maze, with exactly one path between any two
    cells, by randomised depth-first search, then keeps each of its walls
    with probability `density`. density=1 gives the perfect maze, which has
    no loops, so a dragon can always corner the knight in a dead end; lower
    values knock walls down and open up loops; density=0 gives no walls.
    """
    cols = rows if cols is None else cols
    rng = random.Random(seed)
    seen, stack, passages = {(0, 0)}, [(0, 0)], set()
    while stack:
        r, c = stack[-1]
        options = [(r + dr, c + dc) for dr, dc in DIRECTIONS
                   if 0 <= r + dr < rows and 0 <= c + dc < cols
                   and (r + dr, c + dc) not in seen]
        if options:
            nxt = rng.choice(options)
            seen.add(nxt)
            stack.append(nxt)
            passages.add(frozenset(((r, c), nxt)))
        else:
            stack.pop()

    walls = []
    for r in range(rows):
        for c in range(cols):
            for nb in ((r, c + 1), (r + 1, c)):
                if nb[0] < rows and nb[1] < cols:
                    if frozenset(((r, c), nb)) not in passages and rng.random() < density:
                        walls.append(((r, c), nb))
    return walls

In [35]:
"""Rules of the game.

Each turn the knight moves first, then the dragon (on the turns it is
allowed to move, see dragon_every). A move is a step to a neighbouring
open cell, or staying put if allow_stay is on.

The knight wins by reaching the goal. It loses if it and the dragon end
up on the same cell, or if it runs out of turns before reaching the goal.

Both sides are strategies with the same two methods: reset(state), called
once before each game, and next_move(state), called once per turn, which
returns the cell to move to. The dragons are in the next cell, the
knight's base class in the Knight cell.
"""
from __future__ import annotations

import random
from abc import ABC, abstractmethod
from dataclasses import dataclass



class IllegalMove(Exception):
    """A strategy returned a cell it cannot move to."""


@dataclass(frozen=True)
class GameState:
    """Everything a strategy can see on its turn."""
    maze: Maze
    knight: tuple            # (row, col)
    dragon: tuple
    goal: tuple              # the cell to reach
    turn: int                # turns played so far
    max_turns: int
    dragon_every: int        # the dragon moves once every this many turns
    allow_stay: bool
    history: tuple           # (knight, dragon) cells after each turn, starting position first

    def moves_from(self, cell):
        """Legal moves from `cell`: its open neighbours, plus `cell` itself
        if staying put is allowed (or if there is nowhere else to go)."""
        moves = self.maze.neighbours(cell)
        if self.allow_stay or not moves:
            moves.append(cell)
        return moves

    def dragon_acts_on(self, turn):
        """True if the dragon moves on the given turn (always, when dragon_every is 1)."""
        return turn % self.dragon_every == self.dragon_every - 1


@dataclass
class GameResult:
    outcome: str     # "won" or "lost", from the knight's point of view
    reason: str
    turns: int
    history: list    # (knight, dragon) cells after each turn


def maze_distances(maze, source):
    """Number of steps from `source` to every cell reachable from it."""
    dist, ring, steps = {source: 0}, {source}, 0
    while ring:
        steps += 1
        ring = {nb for cell in ring for nb in maze.neighbours(cell) if nb not in dist}
        dist.update(dict.fromkeys(ring, steps))
    return dist


# ======================================================================
# The game
# ======================================================================

class Game:
    def __init__(self, maze, knight, dragon, *, knight_start=None, dragon_start=None,
                 goal=None, max_turns=200, dragon_every=1, allow_stay=True):
        """A game between a knight strategy and a dragon strategy.

        Starting cells default to the top-left corner (knight) and the
        centre (dragon); the goal defaults to the bottom-right corner.
        """
        self.maze = maze
        self.knight_strategy, self.dragon_strategy = knight, dragon
        self.knight_start = tuple(knight_start or (0, 0))
        self.dragon_start = tuple(dragon_start or (maze.rows // 2, maze.cols // 2))
        self.goal = tuple(goal or (maze.rows - 1, maze.cols - 1))
        self.max_turns, self.dragon_every, self.allow_stay = max_turns, dragon_every, allow_stay
        self._check_setup()
        self.reset()

    def _check_setup(self):
        for name, cell in (("knight_start", self.knight_start),
                           ("dragon_start", self.dragon_start), ("goal", self.goal)):
            if not self.maze.in_bounds(cell):
                raise ValueError(f"{name} {cell} is outside the {self.maze.rows}x{self.maze.cols} maze")
        if self.knight_start == self.dragon_start:
            raise ValueError("the knight and the dragon can't start on the same cell")
        if self.goal == self.knight_start:
            raise ValueError("the knight can't start on the goal")
        if self.goal not in maze_distances(self.maze, self.knight_start):
            raise ValueError(f"the walls cut the goal {self.goal} off from the knight")
        if self.dragon_every < 1:
            raise ValueError("dragon_every must be at least 1")

    def reset(self):
        """Put both pieces back on their starting cells."""
        self.knight, self.dragon = self.knight_start, self.dragon_start
        self.turn = 0
        self.history = [(self.knight, self.dragon)]
        self.result = None
        self.knight_strategy.reset(self.state)
        self.dragon_strategy.reset(self.state)

    @property
    def state(self):
        return GameState(self.maze, self.knight, self.dragon, self.goal, self.turn,
                         self.max_turns, self.dragon_every, self.allow_stay,
                         tuple(self.history))

    def _ask(self, who, strategy, here):
        state = self.state
        move = strategy.next_move(state)
        try:
            move = tuple(int(x) for x in move)
        except (TypeError, ValueError):
            pass
        legal = state.moves_from(here)
        if move not in legal:
            raise IllegalMove(f"{strategy!r} tried to move the {who} from {here} to "
                              f"{move!r} on turn {self.turn}; legal moves were {legal}")
        return move

    def step(self):
        """Play one turn: the knight moves, then the dragon if it acts this turn.
        Returns the GameResult once the game is over, otherwise None."""
        if self.result:
            return self.result
        outcome = None
        self.knight = self._ask("knight", self.knight_strategy, self.knight)
        if self.knight == self.dragon:
            outcome = ("lost", "the knight walked into the dragon")
        elif self.knight == self.goal:
            outcome = ("won", "the knight reached the goal")
        elif self.state.dragon_acts_on(self.turn):
            self.dragon = self._ask("dragon", self.dragon_strategy, self.dragon)
            if self.dragon == self.knight:
                outcome = ("lost", "the dragon caught the knight")
        self.turn += 1
        self.history.append((self.knight, self.dragon))
        if outcome is None and self.turn >= self.max_turns:
            outcome = ("lost", "the knight ran out of turns")
        if outcome:
            self.result = GameResult(*outcome, self.turn, list(self.history))
        return self.result

    def run(self):
        """Play until the game is over and return the GameResult."""
        while not self.result:
            self.step()
        return self.result

    def __str__(self):
        marks = {self.goal: "G", self.dragon: "D"}
        marks[self.knight] = "X" if self.knight == self.dragon else "K"
        board = self.maze.render(marks, knight_path=[k for k, _ in self.history],
                                 dragon_path=[d for _, d in self.history])
        status = (f"turn {self.turn}/{self.max_turns}   knight {self.knight}   "
                  f"dragon {self.dragon}   ({self.knight_strategy!r} vs {self.dragon_strategy!r})")
        if self.result:
            status += f"\n{self.result.outcome.upper()}: {self.result.reason} ({self.result.turns} turns)"
        return board + "\n" + status

    __repr__ = __str__

In [36]:
"""The dragon's strategies, roughly from easiest to hardest: StaticDragon,
GuardDragon, GreedyDragon and ChaserDragon.

A dragon has the same two methods as a knight: reset(state), called once
before each game, and next_move(state), called on the dragon's turns,
which returns the cell to move to.
"""
from abc import ABC, abstractmethod


class DragonStrategy(ABC):
    def reset(self, state):
        """Called once before each game with the starting state."""

    @abstractmethod
    def next_move(self, state):
        """Return the cell the dragon moves to: one of state.moves_from(state.dragon)."""

    def __repr__(self):
        return type(self).__name__


def _towards(state, target):
    """The dragon's move that brings it closest to `target` through the maze."""
    dist = maze_distances(state.maze, target)
    return min(state.moves_from(state.dragon), key=lambda cell: dist.get(cell, float("inf")))


class StaticDragon(DragonStrategy):
    """Sleeps where it stands."""

    def next_move(self, state):
        moves = state.moves_from(state.dragon)
        return state.dragon if state.dragon in moves else moves[0]


class GreedyDragon(DragonStrategy):
    """Steps towards the knight as the crow flies, ignoring walls, so it
    gets stuck behind them."""

    def next_move(self, state):
        return min(state.moves_from(state.dragon),
                   key=lambda cell: manhattan(cell, state.knight))


class ChaserDragon(DragonStrategy):
    """Follows the shortest path through the maze to the knight."""

    def next_move(self, state):
        return _towards(state, state.knight)


class GuardDragon(DragonStrategy):
    """Guards its lair (by default the cell it starts on). It chases the
    knight only while the knight is within `radius` steps of the lair, and
    otherwise goes back to it."""

    def __init__(self, radius=3, lair=None):
        self.radius, self.lair = radius, lair

    def reset(self, state):
        self._lair = tuple(self.lair) if self.lair is not None else state.dragon

    def next_move(self, state):
        to_lair = maze_distances(state.maze, self._lair)
        intruder = to_lair.get(state.knight, float("inf")) <= self.radius
        return _towards(state, state.knight if intruder else self._lair)


In [37]:
"""Play and evaluate knight vs dragon games.

Set the parameters in the configuration cell below and run it, or call
play() and evaluate() yourself:

    play(CustomKnight, ChaserDragon)                          # watch one game
    play(CustomKnight, GuardDragon, walls="random", seed=3)
    evaluate(CustomKnight, ChaserDragon, games=50, walls="random")

A knight or dragon is a strategy class, or an instance of one. Any setting
in CONFIG can be passed to play() and evaluate() as a keyword argument.
"""
import time
from collections import Counter


CONFIG = {
    "size": 10,            # side of a square maze, or (rows, cols)
    "walls": None,         # None (no walls), "random", or a list of ((r, c), (r, c)) pairs
    "density": 0.8,        # for walls="random": share of walls kept (1 = no loops at all)
    "seed": None,          # random seed for walls="random"
    "knight_start": None,  # None: the top-left corner
    "dragon_start": None,  # None: the centre
    "goal": None,          # None: the bottom-right corner
    "max_turns": 200,
    "dragon_every": 1,     # the dragon moves once every this many turns (2 = half speed)
    "allow_stay": True,    # whether standing still counts as a move
}


def as_strategy(spec, who):
    """A strategy class is instantiated; an instance is used as it is."""
    strategy = spec() if isinstance(spec, type) else spec
    if not hasattr(strategy, "next_move"):
        raise TypeError(f"the {who} must be a strategy class or an instance of one, got {spec!r}")
    return strategy


def make_maze(size=10, walls=None, density=0.8, seed=None):
    rows, cols = (size, size) if isinstance(size, int) else size
    if walls is None:
        return Maze(rows, cols)
    if walls == "random":
        return Maze(rows, cols, random_walls(rows, cols, density, seed))
    if isinstance(walls, str):
        raise ValueError(f"unknown walls {walls!r}: use None, 'random', or a list of wall pairs")
    return Maze(rows, cols, walls)


def make_game(knight, dragon, **settings):
    unknown = set(settings) - set(CONFIG)
    if unknown:
        raise TypeError(f"unknown settings {sorted(unknown)}; the settings are {list(CONFIG)}")
    cfg = {**CONFIG, **settings}
    maze = make_maze(cfg["size"], cfg["walls"], cfg["density"], cfg["seed"])
    return Game(maze, as_strategy(knight, "knight"), as_strategy(dragon, "dragon"),
                knight_start=cfg["knight_start"], dragon_start=cfg["dragon_start"],
                goal=cfg["goal"], max_turns=cfg["max_turns"],
                dragon_every=cfg["dragon_every"], allow_stay=cfg["allow_stay"])


def _drawer(game):
    """A function that redraws the game in place: in the output cell under
    Jupyter, otherwise by clearing the terminal."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
    except ImportError:
        shell = None
    if shell is not None and "IPKernelApp" in shell.config:
        from IPython.display import display
        handle = None

        def draw():
            nonlocal handle
            if handle is None:
                handle = display(game, display_id=True)
            else:
                handle.update(game)
        return draw
    return lambda: print("\033[H\033[2J" + str(game), flush=True)


def play(knight, dragon, delay=0.2, show=True, **settings):
    """Play one game, drawing every turn, and return the GameResult."""
    game = make_game(knight, dragon, **settings)
    if not show:
        return game.run()
    draw = _drawer(game)
    draw()
    while not game.result:
        time.sleep(delay)
        game.step()
        draw()
    return game.result


def evaluate(knight, dragon, games=20, **settings):
    """Play several games without drawing them, print a summary and return
    the results. With walls="random", each game gets its own maze."""
    first_seed = settings.pop("seed", None) or 0
    results, names = [], None
    for i in range(games):
        game = make_game(knight, dragon, seed=first_seed + i, **settings)
        names = names or f"{game.knight_strategy!r} vs {game.dragon_strategy!r}"
        results.append(game.run())
    wins = [r for r in results if r.outcome == "won"]
    print(f"{names}: won {len(wins)} of {games} ({100 * len(wins) / games:.0f}%)")
    for reason, count in Counter(r.reason for r in results).most_common():
        print(f"  {count:4d}  {reason}")
    if wins:
        print(f"  average turns in a win: {sum(r.turns for r in wins) / len(wins):.1f}")
    return results

## Working with the maze 
## Only read the code from here

In [38]:
# The maze from the drawing above: 3 rows, 4 columns, three walls.
## generate a empty maze with default size 10x10
maze = Maze()

## if we want a walled maze, we can use the random_walls() function
## the first argument is the number of the columns,
## density regulates the number of walls: 0 no walls, 1 entirely locked. 

#maze = Maze(10, walls=random_walls(10, density=0.5))


## set up starting positions
knight_position = (0, 0)  
dragon_position = (1, 2)  
goal_position = (8, 8)   

## print the maze iself, with the knight, dragon, and goal marked
print(maze.render({knight_position: "K", 
                   dragon_position: "D", 
                   goal_position: "G"}))

## now we can also print the entire graph as an adiacency list.
print("maze as adiacency list:")
print(maze.adjacency())

## if we want to access the neighbours of a specific cell, 
## we can use the neighbours() method:
print("neighbours of (0, 0):")
print(maze.neighbours((knight_position)))




█████████████████████████████████████████
█ K                                     █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█         D                             █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█                                       █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█                                       █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█                                       █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█                                       █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█                                       █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█                                       █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█                                 G     █
█   ·   ·   ·   ·   ·   ·   ·   ·   ·   █
█                                       █
█████████████████████████████████████████
maze as adiacency list:
{(0, 0): [(1, 0), (0, 1)], (0, 1): [(1, 1), (0, 0), (0, 2)], (0, 2): [(1, 2), (0, 1), (0, 3)],

In [ ]:
### define first the search algorithms here. 

from collections import deque
import heapq

def bfs(maze, knight_position, goal_position):
    
    visited = {knight_position}
    queue = deque([knight_position])
    padre = {}
    cella_corrente = goal_position
    dragon_position = (1,2)

    while queue:
        node = queue.popleft()
        
        for neighbor in maze.neighbours(node):
            if neighbor not in visited and neighbor == goal_position:
                visited.add(neighbor) 
                queue.append(neighbor)
                padre[neighbor] = node  
                queue.clear()
                break

            if neighbor not in visited and neighbor != dragon_position:
                visited.add(neighbor) 
                queue.append(neighbor)
                padre[neighbor] = node  

    if goal_position not in padre:
        return None
                
    while padre[cella_corrente] != knight_position:
        cella_corrente = padre[cella_corrente]

    return cella_corrente
    

from collections import deque
import heapq
import math

def dijkstra(maze, knight_position, goal_position, cost):
#You have two options: 
#precompute the cost of each cell, or remove cost from the arguments and use a weighted graph. 
#You can add infinite cost for the dragon cell, but also add a cost for the nearby cells.

    dist_dal_drago = distances(maze, dragon_position)

    Dist_min = {nodo_corrente: math.inf for nodo_corrente in maze.cells()}
    Tree = {knight_position: None}
    Nodi_visitati = set()
    priority_queue = []

    Dist_min[knight_position] = 0
    heapq.heappush(priority_queue, (0, knight_position))

    while priority_queue:
        dist_nodo_corrente, nodo_corrente = heapq.heappop(priority_queue)

        if nodo_corrente == goal_position:
            break
        if nodo_corrente in Nodi_visitati:
            continue

    Nodi_visitati.add(nodo_corrente)

    for neighbor in maze.neighbours(nodo_corrente):
        if neighbor == dragon_position:
            continue

    # Aumenta il costo se la cella è entro 3 passi dal drago[cite: 2]
    dist_drago = dist_dal_drago.get(neighbor, math.inf)
    cost = 1000 if dist_drago <= 3 else 1

    if Dist_min[neighbor] == math.inf or Dist_min[nodo_corrente] + cost < Dist_min[neighbor]:
        Dist_min[neighbor] = Dist_min[nodo_corrente] + cost
        heapq.heappush(priority_queue, (Dist_min[neighbor], neighbor))
        Tree[neighbor] = nodo_corrente

    if goal_position not in Tree:
        return knight_position

    cella_corrente = goal_position
    while Tree[cella_corrente] != knight_position:
        cella_corrente = Tree[cella_corrente]

    return cella_corrente

def distances(maze, source):
#This function can be useful if you want to find what are the cells in radius from the dragon(=source)

    dist, area, passi = {source: 0}, {source}, 0
    while area:
        passi += 1
        area = {neighbours for cell in area for neighbours in maze.neighbours(cell) if neighbours not in dist}
        dist.update(dict.fromkeys(area,passi))

    return dist


In [ ]:
"""The knight's strategy. Subclass KnightStrategy to write your own knight.

The knight's maze works similarly to the example maze. 
The information regarding the maze are in the knight's state. 
    
    state.knight, state.dragon, state.goal --> return tuples (row, col), with the positions.    
    state.maze --> is the same maze object as the one above. 
                   you can use the state.maze.neighbours(cell) to get the neighbours of a cell, 
                   state.maze.is_open(a, b) to check if there is a wall between two cells.
                   state.maze.adjacency() gets the adjacency list.
    state.moves_from(cell)        legal moves from a cell (also the cell itself, since the knight can stay put)

    those are ignorable:
    state.turn, state.max_turns   turns played so far, and the limit
    state.dragon_acts_on(turn)    whether the dragon moves on a given turn
    state.history   

The knight is himself an agent. 
Of the knight, you have to implement the next_move() method. 
Each turn the game calls next_move(state).
next_move returns a tuple (row, col), corresponding to the cell to move to, which must be one of state.moves_from(state.knight).
each turn the knight state is then updated, with the new positions of the knight and the dragon. 


    class CustomKnight(KnightStrategy):
        def reset(self, state):
            ...  # optional: runs once before each game
                 ## if you put in the knight state the adjacency list, 
                 ## or something else, you can use reset across games to restore the knight. 

        def next_move(self, state)-> tuple(int, int):
            ...  # return the cell to move to

Implement there the strategy to move to the treasure, happy hunting! 


"""
from abc import ABC, abstractmethod


class KnightStrategy(ABC):
    def reset(self, state):
        """Called once before each game with the starting state. Override it
        to precompute things, or to clear anything kept from a previous game."""

    @abstractmethod
    def next_move(self, state):
        """Return the cell to move the knight to: one of state.moves_from(state.knight)."""

    def __repr__(self):
        return type(self).__name__


# ======================================================================
# An example knight: just rushing to the goal. 
# ======================================================================

class GreedyKnight(KnightStrategy):
    """Heads for the goal, ignoring walls and the dragon.
    Greed incarnate: it gets stuck behind walls and walks into the dragon."""

    def next_move(self, state):
        moves = state.moves_from(state.knight)
        return min(moves, key=lambda cell: manhattan(cell, state.goal))


In [41]:
# Here, you can create a custom knight that subclasses KnightStrategy,
# and implements a movement algorithm, able to beat the dragon. 

from collections import deque
import heapq

AI_ASSISTANCE = "none"   # which AI tools you used, and for what

## Try to implement first BFS, then Dijkstra, 
## Then, if you have time, try to beat the harder dragons,
## With new intuitions. 

class CustomKnight(KnightStrategy):

    def next_move(self, state):
        moves = state.moves_from(state.knight)
        return min(moves, key=lambda cell: manhattan(cell, state.goal))


In [ ]:
# Configuration: edit the values below and run this cell.
## To test the various levels change the file, or write a script that sets the parameters. 
## Ideally, save the results in a dictionary. 
## level_results: dict = {level_number: float = winrate: float}


## Level 1: 
## GAMES = 50
## walls = 'random'

## Level 2: 
## GAMES = 50
## walls = 'random'
## DRAGON = GuardDragon

## Level 2.5:
## GAMES = 50
## walls = 'random'
## DRAGON = GreedyDragon

## Level 3:
## GAMES = 50 
## walls = 'random'
## DRAGON = ChaserDragon
## density = 0.5
## "dragon_every": 1 or 2 


KNIGHT = CustomKnight      # your knight from the cell above, or GreedyKnight for the baseline
DRAGON = StaticDragon  # StaticDragon, GuardDragon, GreedyDragon or ChaserDragon
GAMES = None           # None: watch one game; a number: play that many games silently and summarise
DELAY = 0.2            # seconds between turns when watching a game

SETTINGS = {
    "size": 10,            # side of a square maze, or (rows, cols), e.g. (8, 12)
    "walls": 'None',         # None (no walls), "random", or a list of ((r, c), (r, c)) pairs
    "density": 0.8,        # for walls="random": share of walls kept (1 = no loops at all)
    "seed": None,          # random seed for walls="random"
    "knight_start": None,  # (row, col); None: the top-left corner
    "dragon_start": None,  # (row, col); None: the centre
    "goal": None,          # (row, col); None: the bottom-right corner
    "max_turns": 200,
    "dragon_every": 1,     # the dragon moves once every this many turns (2 = half speed)
    "allow_stay": True,    # whether standing still counts as a move
}

if GAMES:
    evaluate(KNIGHT, DRAGON, games=GAMES, **SETTINGS)
else:
    play(KNIGHT, DRAGON, delay=DELAY, **SETTINGS)

█████████████████████████████████████████
█ →   ↓ █           █               █   █
█████   █████   █████   █████   █   █   █
█     ↓ █       █       █       █       █
█   ·   █   █   █   █████   █   █████   █
█     ↓ █   █   █           █           █
█   █   █   █████████████   █   ·   █   █
█   █ →   ↓         █       █       █   █
█   █████   █   █████   █████   █   █   █
█       █ ↓ █                   █       █
█████   █   █   ·   █   █████████████   █
█       █ K █       █ D                 █
█   █████████   ·   █   █   ·   █   ·   █
█   █       █           █       █       █
█   █   ·   █   █████   █████   █   █████
█   █           █                       █
█   █   █   ·   █   █████   █   █████   █
█       █                   █   █       █
█████████   █████████████████   █   ·   █
█                               █     G █
█████████████████████████████████████████
turn 111/200   knight (5, 2)   dragon (5, 5)   (CustomKnight vs StaticDragon)